In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import wandb
from datasets import Dataset
from transformers import AutoTokenizer
from finetune import train_eval_model
from experiment import preprocess_text

2026-05-23 23:46:06.122570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779569166.148869  270798 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779569166.156684  270798 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779569166.181666  270798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779569166.181688  270798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779569166.181691  270798 computation_placer.cc:177] computation placer alr

In [2]:
models_config = [
    ("sentence-transformers/all-MiniLM-L6-v2", "outputs/emotion-classifier-minilm")
]
label_config = [
    {0: 'neutral', 0.5: 'present'}, 
    {0: 'neutral', 0.5: 'medium', 1.5: 'high'},
    {0: 'neutral', 0.5: 'moderate', 1.0: 'high', 1.5: 'strong', 2.0: 'extreme'}
]

In [3]:
def preprocess(example):
    return {'text': preprocess_text(example['text']), 'labels': example['labels']}

def load_emotionality_dataset(tokenizer, label_bounds = {0: 'present'}):
    DATA_PATH = Path("../data/2026-05-21/corpus_emotions.json")
    data = pd.read_json(DATA_PATH)
    labels = pd.cut(
        data['emotionality'], 
        bins=list(label_bounds.keys()) + [np.max(data['emotionality'])+1],
        labels=list(label_bounds.values()),
        right=False
    ).astype('object')
    data = Dataset.from_dict({'text': data['content'], 'labels': labels})
    data = data.map(preprocess)
    labels_map = {label: ind for ind, label in enumerate(set(data['labels']))}

    def tokenize(example):
        labels = list(map(lambda m: labels_map.get(m), example['labels']))
        example = tokenizer(example["text"], padding="max_length", truncation=True)
        example['labels'] = labels
        return example

    final_dst = data.map(tokenize, batched=True, remove_columns=['text'])
    final_dst.set_format(type="torch")
    return final_dst, labels_map

In [ ]:
for model_name, output_dir in models_config:
    for ind, label_bounds in enumerate(label_config):
        print(f"Training model for emotion detection with model {model_name} using discretization to {len(label_bounds)} classes")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        ds, labels_map = load_emotionality_dataset(tokenizer, label_bounds=label_bounds)
        run = wandb.init(project="emotion-detection-lt", name=f"{model_name}-{len(label_bounds)}class-{ind}")
        train_eval_model(
            model_name=model_name,
            dataset=ds,
            labels_map=labels_map,
            output_dir=f"{output_dir}-{ind}", 
            batch_size=16, 
            num_epochs=20, 
            eval_batch_size=16,
            report_to=['tensorboard', 'wandb']
        )

Training model for emotion detection with model sentence-transformers/all-MiniLM-L6-v2 using discretization to 2 classes


Map:   0%|          | 0/2660 [00:00<?, ? examples/s]

Map:   0%|          | 0/2660 [00:00<?, ? examples/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/paulius/.netrc.
wandb: Currently logged in as: danpaulius (danpaulius-ktu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Stringifying the column:   0%|          | 0/2660 [00:00<?, ? examples/s]

...

loading file vocab.txt from cache at /home/paulius/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/vocab.txt
loading file tokenizer.json from cache at /home/paulius/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at /home/paulius/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/special_tokens_map.json
loading file tokenizer_config.json from cache at /home/paulius/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json
loading file chat_template.jinja from cache at None
loading configuration file config.json from cache at /home/paulius/.cache/huggingface/hub/models--senten

Stringifying the column:   0%|          | 0/2660 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/2660 [00:00<?, ? examples/s]

loading configuration file config.json from cache at /home/paulius/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json
Model config BertConfig {
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 384,
  "id2label": {
    "0": "high",
    "1": "neutral",
    "2": "medium"
  },
  "initializer_range": 0.02,
  "intermediate_size": 1536,
  "label2id": {
    "high": 0,
    "medium": 2,
    "neutral": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 6,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.3",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

loading weights file model.safetensors 

Epoch,Training Loss,Validation Loss
